# Assignment 02 - Project: Deep Research with LangGraph - Scope
This notebook implements **Module 2: Scope - User Clarification and Brief Generation**. Before conducting deep research, our agent asks clarifying questions to narrow down the topic scope, waits for the user's answers, and compiles a structured Research Brief.

In [1]:
import sys
import os
from dotenv import load_dotenv

# Allow importing mock_llm from the parent directory
sys.path.append(os.path.abspath(".."))
from mock_llm import get_llm

load_dotenv()
print("Environment loaded successfully.")

Environment loaded successfully.


In [2]:
from typing import List, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Define the state schema for scoping
class ScopingState(TypedDict):
    topic: str
    questions: List[str]
    answers: List[str]
    brief: str

In [3]:
def generate_clarification_questions(state: ScopingState):
    """Invokes the LLM to draft clarification questions about the research topic."""
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = f"Draft 3 clarifying questions in JSON format under key 'questions' to narrow down research on: {state['topic']}"
    
    # Invoke the LLM
    res = llm.invoke(prompt)
    try:
        data = json.loads(res.content)
        questions = data.get("questions", [])
    except:
        # Fallback if LLM didn't return valid JSON
        questions = [
            "What is the primary target audience?",
            "Should we focus on symmetric or asymmetric algorithms?",
            "What is the expected depth of the final technical migration timeline?"
        ]
    return {"questions": questions}

In [4]:
def generate_brief(state: ScopingState):
    """Combines the topic, questions, and answers to compile the final Research Brief."""
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    Q_and_A = "\n".join([f"Q: {q}\nA: {a}" for q, a in zip(state["questions"], state["answers"])])
    prompt = f"""Compile a structured Research Brief for topic: {state['topic']}
Using these clarifying Q&A:
{Q_and_A}
Structure it with headings for Topic, Focus, Target Audience, and Depth of Research."""
    
    res = llm.invoke(prompt)
    return {"brief": res.content}

In [5]:
# Build the scoping graph
builder = StateGraph(ScopingState)
builder.add_node("ask_questions", generate_clarification_questions)
builder.add_node("generate_brief", generate_brief)

builder.add_edge(START, "ask_questions")
builder.add_edge("ask_questions", "generate_brief")
builder.add_edge("generate_brief", END)

# Enable checkpointer to support interrupt_before
memory = MemorySaver()
graph = builder.compile(checkpointer=memory, interrupt_before=["generate_brief"])

In [6]:
# Print the graph architecture
try:
    print(graph.get_graph().draw_ascii())
except Exception as e:
    print("Could not draw graph:", e)

  +-----------+    
  | __start__ |    
  +-----------+    
         *         
         *         
         *         
+---------------+  
| ask_questions |  
+---------------+  
         *         
         *         
         *         
+----------------+ 
| generate_brief | 
+----------------+ 
         *         
         *         
         *         
    +---------+    
    | __end__ |    
    +---------+    


In [7]:
import json

# Step 1: Start the graph run. It will ask questions and interrupt before brief generation.
thread_config = {"configurable": {"thread_id": "thread-1"}}
initial_state = {"topic": "quantum computing threats to cryptography", "answers": []}

print("--- Executing Step 1: Generating Questions ---")
for event in graph.stream(initial_state, thread_config, stream_mode="values"):
    if "questions" in event and event["questions"]:
        print("\nGenerated Clarification Questions:")
        for idx, q in enumerate(event["questions"], 1):
            print(f"{idx}. {q}")

--- Executing Step 1: Generating Questions ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---



Generated Clarification Questions:
1. What is the primary target audience?
2. Should we focus on symmetric or asymmetric algorithms?
3. What is the expected depth of the final technical migration timeline?


In [8]:
# Step 2: Feed answers to the checkpointer state and resume
user_answers = [
    "Focus on post-quantum migration for enterprise asymmetric key algorithms.",
    "Provide standard technical recommendations according to NIST.",
    "The depth should be an executive summary with a 5-year timeline."
]

print("--- Executing Step 2: Updating Answers & Resuming Graph ---")
graph.update_state(thread_config, {"answers": user_answers})

for event in graph.stream(None, thread_config, stream_mode="values"):
    if "brief" in event and event["brief"]:
        print("\n--- Generated Research Brief ---")
        print(event["brief"])

--- Executing Step 2: Updating Answers & Resuming Graph ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---



--- Generated Research Brief ---
**Research Brief**

---

### 1. Topic  
**Quantum‑Computing Threats to Cryptography**

---

### 2. Focus  
- **Post‑Quantum Migration** for enterprise **asymmetric key** algorithms (public‑key encryption, digital signatures, key‑exchange).  
- Align recommendations with the **NIST Post‑Quantum Cryptography (PQC) Standardization Process** (Rounds 1‑3 and final selections).  
- Provide a **practical 5‑year migration roadmap** that senior leadership can present to boards, audit committees, and IT steering groups.

---

### 3. Target Audience  
- **Enterprise security executives** (CISO, VP‑Security, Chief Risk Officer).  
- **IT architects & engineering leads** responsible for PKI, TLS/SSL, VPN, code signing, and secure email.  
- **Risk and compliance officers** who must map migration to regulatory frameworks (PCI‑DSS, GDPR, HIPAA, etc.).  

The brief is intentionally high‑level (executive summary) but includes enough technical detail for implementation 